# Equal Earth Projection Mapping Pipeline

This notebook implements the **Equal Earth map projection** using custom mathematical transformations. It allows you to ingest geographic coordinates (latitude and longitude) from Google Drive via CSV, Excel, or Google Sheets, project them accurately, and export professional visualizations directly back to your Drive.

## Background & Mathematical Breakdown

### The Problem with Mercator Maps
For centuries, the **Mercator projection** has been the standard for web maps and navigation because it preserves angles and shapes locally. However, it drastically distorts the **relative size** of landmasses as you move further from the equator. For example, on a Mercator map, Greenland appears roughly the same size as Africa, whereas Africa is actually **14 times larger** in reality.

### The Equal Earth Advantage
Developed in 2018 by Bojan Šavrič, Tom Patterson, and Bernhard Jenny, the **Equal Earth projection** is a pseudocylindrical, **equal-area projection**. It ensures that the relative sizes of all regions on the map match their actual sizes on the globe, while maintaining a visually pleasing, familiar aesthetic inspired by the Robinson projection.

### The Formula Clarified
The forward projection transforms spherical coordinates (Longitude $\lambda$ and Latitude $\varphi$) into Cartesian coordinates ($x, y$):

1. **Parametric Latitude ($\theta$):**
$$\sin \theta = \frac{\sqrt{3}}{2} \cdot \sin \varphi$$

2. **X-Coordinate (Horizontal Position):**
$$x = \frac{2 \cdot \sqrt{3} \cdot \lambda \cdot \cos \theta}{3 \cdot (A_1 + 3 \cdot A_2 \cdot \theta^2 + \theta^6 \cdot (7 \cdot A_3 + 9 \cdot A_4 \cdot \theta^2))}$$

3. **Y-Coordinate (Vertical Position):**
$$y = \theta \cdot (A_1 + A_2 \cdot \theta^2 + \theta^6 \cdot (A_3 + A_4 \cdot \theta^2))$$

> **Note on Constants:** According to the official mathematical specification of the Equal Earth projection, $A_2$ must be negative ($A_2 = -0.081106$) to correctly adjust the curvature. You can see this implemented in J3 here as well: https://github.com/d3/d3-geo/blob/main/src/projection/equalEarth.js.

The full set of official constants is:
* $A_1 = 1.340264$
* $A_2 = -0.081106$
* $A_3 = 0.000893$
* $A_4 = 0.003796$

## Step 1: Environment Setup & Dependencies

*Run this cell to install required geospatial tools and mount your Google Drive.*

In [ ]:
# Install specialized geospatial packages if not pre-installed in Colab
!pip install -q geopandas shapely

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
from google.colab import drive, auth

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Authenticate for Google Sheets access
auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

print("Setup complete! Google Drive mounted and libraries imported.")

## Step 2: Defining the Equal Earth Projection Functions

*This cell implements the exact formulas from the image using NumPy to support vectorized operations on entire datasets at once.*

In [ ]:
def to_equal_earth(lon_deg, lat_deg, lon_0_deg=0):
    """
    Transforms Longitude and Latitude degrees into Equal Earth X and Y coordinates.

    Parameters:
    - lon_deg, lat_deg: Coordinate arrays or scalars in degrees
    - lon_0_deg: Central meridian for your map (default is 0)
    """
    # Convert degrees to radians
    lam = np.radians(lon_deg - lon_0_deg)
    phi = np.radians(lat_deg)

    # Constants
    A1 = 1.340264
    A2 = -0.081106
    A3 = 0.000893
    A4 = 0.003796

    # 1. Calculate Parametric Latitude (theta)
    sin_theta = (np.sqrt(3) / 2.0) * np.sin(phi)
    # Clip values to avoid floating-point errors outside [-1, 1]
    sin_theta = np.clip(sin_theta, -1.0, 1.0)
    theta = np.arcsin(sin_theta)

    theta_sq = theta ** 2
    theta_six = theta ** 6
    cos_theta = np.cos(theta)

    # 2. Calculate X
    num_x = 2.0 * np.sqrt(3) * lam * cos_theta
    den_x = 3.0 * (A1 + 3.0 * A2 * theta_sq + theta_six * (7.0 * A3 + 9.0 * A4 * theta_sq))
    x = num_x / den_x

    # 3. Calculate Y
    y = theta * (A1 + A2 * theta_sq + theta_six * (A3 + A4 * theta_sq))

    return x, y

## Step 3: Unified Data Ingestion Pipeline

*Configure your input source here. Change the paths below to match the location of your files on Google Drive.*

In [ ]:
# --- CONFIGURATION BOX ---
INPUT_TYPE = "SHAPEFILE"  # Options: "CSV", "EXCEL", "GSHEET", "SHAPEFILE"
DRIVE_FOLDER = "/content/drive/MyDrive/EqualEarthMapping/"  # Output and working directory

# Example File paths (Adjust these to match your actual files)
CSV_PATH = DRIVE_FOLDER + "my_coordinates.csv"
EXCEL_PATH = DRIVE_FOLDER + "my_coordinates.xlsx"
GSHEET_URL = "https://docs.google.com/spreadsheets/d/your-sheet-id-here/edit"
# -------------------------

# Create the folder if it doesn't exist
os.makedirs(DRIVE_FOLDER, exist_ok=True)

def load_data():
    if INPUT_TYPE == "CSV":
        df = pd.read_csv(CSV_PATH)
        return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude))

    elif INPUT_TYPE == "EXCEL":
        df = pd.read_excel(EXCEL_PATH)
        return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude))

    elif INPUT_TYPE == "GSHEET":
        wb = gc.open_by_url(GSHEET_URL)
        sheet = wb.get_sheet_by_index(0)
        data = sheet.get_all_records()
        df = pd.DataFrame(data)
        return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude))

    elif INPUT_TYPE == "SHAPEFILE":
        # Automatically downloads a baseline world map grid if no data is provided
        world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
        return world

gdf = load_data()
print(f"Successfully loaded data. Structure contains columns: {list(gdf.columns)}")

## Step 4: Map Generation & Region Customization

*This script transforms geometries, applies geographic bounds to isolate regions (World, Europe, Africa, etc.), and sets up visualization options.*

In [ ]:
# --- VIEWPORT CONFIGURATION ---
# Change REGION to "WORLD", "AFRICA", "EUROPE", or "CUSTOM"
REGION = "WORLD"

# Custom bounding box boundaries (ignored if REGION is not "CUSTOM")
MIN_LON, MAX_LON = -20, 50
MIN_LAT, MAX_LAT = -35, 40
# ------------------------------

def filter_and_project(gdf_input, region_type):
    # Apply bounding box filters based on selection
    if region_type == "AFRICA":
        box = Polygon([(-20, -35), (55, -35), (55, 40), (-20, 40)])
        gdf_filtered = gdf_input[gdf_input.intersects(box)].copy()
    elif region_type == "EUROPE":
        box = Polygon([(-25, 35), (45, 35), (45, 75), (-25, 75)])
        gdf_filtered = gdf_input[gdf_input.intersects(box)].copy()
    elif region_type == "CUSTOM":
        box = Polygon([(MIN_LON, MIN_LAT), (MAX_LON, MIN_LAT), (MAX_LON, MAX_LAT), (-MIN_LON, MAX_LAT)])
        gdf_filtered = gdf_input[gdf_input.intersects(box)].copy()
    else:
        gdf_filtered = gdf_input.copy()

    # Apply Equal Earth conversion to all geometries
    projected_features = []
    for geom in gdf_filtered.geometry:
        if geom.geom_type == 'Point':
            nx, ny = to_equal_earth(geom.x, geom.y)
            projected_features.append(Point(nx, ny))
        elif geom.geom_type == 'Polygon':
            ex, ey = to_equal_earth(*geom.exterior.coords.xy)
            projected_features.append(Polygon(zip(ex, ey)))
        elif geom.geom_type == 'MultiPolygon':
            poly_list = []
            for poly in geom.geoms:
                ex, ey = to_equal_earth(*poly.exterior.coords.xy)
                poly_list.append(Polygon(zip(ex, ey)))
            projected_features.append(gpd.GeoSeries(poly_list).unary_union)

    gdf_projected = gpd.GeoDataFrame(gdf_filtered, geometry=projected_features)
    return gdf_projected

# Process geometries
gdf_ee = filter_and_project(gdf, REGION)
print(f"Geometries successfully projected using Equal Earth for target view: {REGION}")

## Step 5: Render Engine and Multi-Format Exporter

*This block handles the visualization layout and exports high-quality image assets and vectorized geometry back to your Google Drive.*

In [ ]:
# --- RENDER OPTIONS ---
OUTPUT_FILENAME = f"equal_earth_{REGION.lower()}_map"
MAP_THEME = "plasma"  # Options: 'viridis', 'plasma', 'magma', 'Blues', etc.
# ----------------------

fig, ax = plt.subplots(figsize=(14, 8), dpi=300)
ax.set_aspect('equal')

# Plot options based on data types loaded
if 'pop_est' in gdf_ee.columns:
    # If using built-in world dataset, generate an authalic choropleth map
    gdf_ee.plot(column='pop_est', cmap=MAP_THEME, legend=True,
                legend_kwds={'label': "Population Estimate", 'orientation': "horizontal"}, ax=ax)
else:
    # If custom user points or tracks are loaded
    gdf_ee.plot(cmap=MAP_THEME, ax=ax, markersize=15, alpha=0.8)

ax.set_title(f"Equal Earth Map Projection - Focus Region: {REGION}", fontsize=16, pad=20)
ax.axis('off') # Clean presentation style without outer boundary axes

# Define output paths
png_export_path = os.path.join(DRIVE_FOLDER, f"{OUTPUT_FILENAME}.png")
pdf_export_path = os.path.join(DRIVE_FOLDER, f"{OUTPUT_FILENAME}.pdf")
geojson_export_path = os.path.join(DRIVE_FOLDER, f"{OUTPUT_FILENAME}.geojson")

# Save outputs directly to Google Drive
plt.savefig(png_export_path, bbox_inches='tight')
plt.savefig(pdf_export_path, bbox_inches='tight')
try:
    gdf_ee.to_file(geojson_export_path, driver="GeoJSON")
    print(f"Vector Data exported: {geojson_export_path}")
except Exception as e:
    print(f"GeoJSON Vector skip: Data structure requires cleaning ({e})")

plt.show()

print(f"\n Success! All visualizations exported safely to your Drive folder:")
print(f"Raster Image: {png_export_path}")
print(f"Document Presentation: {pdf_export_path}")